# RAG on a codebase
### Retreival Augmented Generation

Thank you Ben Clavié for getting me started on this path. 

https://parlance-labs.com/education/rag/ben.html

![rag-tree.webp](rag-tree.webp)


- RAG is Retreival Augmented Generation. 
- It just means 'provide relevant context'
- It works by 
1. creating an embedding from a prompt
2. creating embeddings from sections of a document
3. finding the cosine similarity between the prompt and each section of the document.
4. providing those most relevant sections of the document to a generative model to generate an answer.

-Once the paragraph with the hightest cosine similarity to the prompt is found, the top 3 sentences are fed into a generative model to generate an answer. 

This project aims to make a RAG system which works on a github repository containing python code.

In [1]:
# %pip install -U sentence-transformers
# %pip install wikipedia-api
# %pip install claudette
# %pip install ollama
# !ollama pull deepseek-r1:1.5b
# %pip install llama-index

# %pip install llama-index-embeddings-huggingface
# %pip install llama-index-embeddings-instructor

# !pip install 'tree-sitter<0.22.0'
# !pip install tree_sitter_languages==1.5.1

In [2]:
from sentence_transformers import SentenceTransformer
from wikipediaapi import Wikipedia
from claudette import Chat, models
import re
import ollama
import numpy as np

from llama_index.core import Document

from pathlib import Path
import tempfile
import os
from llama_index.core import Document

/Users/mikeg/miniforge3/envs/ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


I got the model list from here
https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

In [3]:
model = SentenceTransformer('Alibaba-NLP/gte-base-en-v1.5', trust_remote_code=True)

In [4]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False}) with Transformer model: NewModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

This model has a context length of 8192 tokens, and an embedding size of 768.

### Fetch some repo text content


In [5]:
with open ('repomix-output-plain.txt') as f:
    text = f.read()

### Choose which chunks to split the document into

- I chose to split the document into files and store them in a dictionary with filenames as keys and the text as values.

In [6]:
def split_codebase(text):
    pattern = r"=+\nFile: (.*?)\n=+\n"
    parts = re.split(pattern, text)
    
    # parts[0] is any header text before the first match (probably empty)
    files = {}
    for i in range(1, len(parts), 2):
        filename = parts[i]
        content = parts[i+1]
        files[filename] = content.strip()
    return files

splits = split_codebase(text)
python_files = {k: v for k, v in splits.items() if k.endswith('.py')}

In [7]:
lengths = {k: len(v) for k, v in splits.items()}
# for k, l in sorted(lengths.items(), key=lambda x: -x[1]):
#     print(f"{k}: {l} chars")

py_lengths = {k: len(v) for k, v in python_files.items()}
for k, l in sorted(py_lengths.items(), key=lambda x: -x[1]):
    print(f"{k}: {l} chars")

opensoundscape/ml/cnn.py: 109613 chars
opensoundscape/audio.py: 79607 chars
opensoundscape/annotations.py: 75706 chars
tests/test_annotations.py: 39533 chars
tests/test_audio.py: 38373 chars
tests/test_cnn.py: 36772 chars
opensoundscape/spectrogram.py: 34028 chars
opensoundscape/preprocess/preprocessors.py: 32976 chars
opensoundscape/localization/synchronized_recorder_array.py: 28718 chars
tests/test_localization.py: 28123 chars
opensoundscape/ml/cnn_architectures.py: 26582 chars
opensoundscape/signal_processing.py: 24265 chars
opensoundscape/localization/spatial_event.py: 22610 chars
opensoundscape/ml/lightning.py: 21840 chars
opensoundscape/localization/audiomoth_sync.py: 19738 chars
opensoundscape/localization/localization_algorithms.py: 16644 chars
opensoundscape/utils.py: 14909 chars
opensoundscape/preprocess/actions.py: 14446 chars
opensoundscape/preprocess/overlay.py: 13876 chars
tests/test_signal_processing.py: 11908 chars
opensoundscape/preprocess/action_functions.py: 11645 ch

### As a first test, just use a couple of the .py files in the repo

In [8]:
py_sample = {k: python_files[k] for k in ["opensoundscape/preprocess/utils.py", "opensoundscape/audio.py"]}

In [9]:
print(py_sample["opensoundscape/audio.py"])
len(py_sample["opensoundscape/audio.py"])

#!/usr/bin/env python3
""" audio.py: Utilities for loading and modifying Audio objects


**Note: Out-of-place operations**

Functions that modify Audio (and Spectrogram) objects are "out of place",
meaning that they return a new Audio object instead of modifying the
original object. This means that running a line
```
audio_object.resample(22050) # WRONG!
```
will **not** change the sample rate of `audio_object`!
If your goal was to overwrite `audio_object` with the new,
resampled audio, you would instead write
```
audio_object = audio_object.resample(22050)
```

"""
import warnings
import datetime
from pathlib import Path
import json
import io
import urllib
import os

import numpy as np
import scipy

import librosa
import librosa.core.audio
import soundfile
import IPython.display
from aru_metadata_parser.parse import parse_audiomoth_metadata
from aru_metadata_parser.utils import load_metadata
import noisereduce

import opensoundscape
from opensoundscape.utils import generate_clip_times

79607

In [10]:
python_files = py_sample

In [11]:

documents = [
    # Document(text=content, metadata={"source": filename, "file_ext": Path(filename).suffix})
    Document(text=content, metadata={"source": filename})
    for filename, content in python_files.items()
]

In [12]:
documents[0].metadata

{'source': 'opensoundscape/preprocess/utils.py'}

### Make embeddings from the split documents


In [13]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
# embed_model = GTEEmbedding("Alibaba-NLP/gte-base-en-v1.5")

index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

### optionally chunk manually 

In [17]:
from llama_index.core.node_parser import CodeSplitter
splitter = CodeSplitter(language="python", chunk_lines=1)
all_nodes = splitter.get_nodes_from_documents(documents, show_progress=True)

Parsing nodes: 100%|██████████| 2/2 [00:00<00:00, 84.67it/s]


In [20]:
from llama_index.core.settings import Settings
Settings.llm = None
query_engine = index.as_query_engine()
# nodes = list(index.docstore.docs.values())

LLM is explicitly disabled. Using MockLLM.


In [22]:
print(all_nodes[4].text)
print("++++++++++++")
print(all_nodes[5].text)

class Audio:
++++++++++++
"""Container for audio samples

    Initialization requires sample array. To load audio file, use
    `Audio.from_file()`

    Initializing an `Audio` object directly requires the specification of the
    sample rate. Use `Audio.from_file` or `Audio.from_bytesio` with
    `sample_rate=None` to use a native sampling rate.

    Args:
        samples (np.array):     The audio samples
        sample_rate (integer):  The sampling rate for the audio samples
        resample_type (str):    The resampling method to use [default: "soxr_hq"]

    Returns:
        An initialized `Audio` object
    """

    __slots__ = ("samples", "sample_rate", "resample_type", "metadata")


The code splitting isn't working as expected: the class definition is in a different node than the rest of the class. 

Using a hierarchical splitter might work better, but I'll try this first.

Another option would be to use 

```python
splitter = CodeSplitter(language="python", chunk_lines=None)
```

In [46]:
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

from llama_index.core.settings import Settings
# Settings.llm = None

query_engine = index.as_query_engine()

In [47]:
response = query_engine.query("How do i apply a lowpass filter to an audio file?")
print(response)

Context information is below.
---------------------
source: opensoundscape/audio.py

def highpass_filter(signal, cutoff_f, sample_rate, order=9):
    """perform a butterworth highpass filter on a discrete time signal
    using scipy.signal's butter and sosfiltfilt (phase-preserving filtering)

    Args:
        signal: discrete time signal (audio samples, list of float)
        cutoff_f: -3db point for highpass filter (Hz)
        sample_rate: samples per second (Hz)
        order: higher values -> steeper dropoff [default: 9]

    Returns:
        filtered time signal
    """
    nyq = 0.5 * sample_rate
    cut = cutoff_f / nyq
    sos = scipy.signal.butter(order, cut, analog=False, btype="highpass", output="sos")
    return scipy.signal.sosfiltfilt(sos, signal)


def bandpass_filter(signal, low_f, high_f, sample_rate, order=9):
    """perform a butterworth bandpass filter on a discrete time signal
    using scipy.signal's butter and sosfiltfilt (phase-preserving filtering)

    Args:

# Running DeepSeek locally without sending data to the cloud
There are plenty of cases where querying a third party server might give away sensitive information.

Since __DeepSeek__ was released it has become much easier to run this process locally. You'll just need to install ollama and then pull the relevant deepseek model, then you can query the LLM without even needing the internet. 

In [60]:
llm = 'deepseek-r1:1.5b'
chat = ollama.chat(model=llm, messages=[{'role': 'user', 'content':  f"{response}"}])
answer = (chat['message']['content'])

# remove the reasoning part leaving just deepseek's answer
cleaned_text = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL)

In [41]:
print(cleaned_text.strip())

To apply a lowpass filter to an audio file:

1. **Create an Audio instance**: 
   - Use your audio data to create an `Audio` object. For example:
     ```python
     import Audio class as AudioInstance
     audio_instance = AudioInstance.Audio()
     ```

2. **Apply the lowpass filter**:
   - Call the `lowpass` method with appropriate parameters.
     ```python
     audio_instance.lowpass(cutoff_f=5000, order=2)
     ```

3. **Adjust sample rate (if needed)**: 
   - Ensure your data is sampled at a rate suitable for the filter.

The lowpass filter will attenuate frequencies above `cutoff_f`, effectively smoothing out high-pitched sounds while preserving lower ones.


# Finetuning deepseek on the codebase

text

In [55]:
! pip uninstall xformers
! pip install unsloth --no-deps
! pip install torch transformers peft accelerate bitsandbytes datasets trl


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [57]:
import unsloth

/var/folders/hy/36n7jr6x3h74ddx507dc12_00000gn/T/ipykernel_2329/3279665559.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


NotImplementedError: Unsloth: No NVIDIA GPU found? Unsloth currently only supports GPUs!

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import TrainingArguments
from datasets import Dataset

# === 1. Load model (DeepSeek 1.5B base)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "deepseek-ai/deepseek-coder-1.5b-base",
    max_seq_length = 2048,
    dtype = "auto",           # or "float32" if needed
    load_in_4bit = True
)

# === 2. Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"],
    bias = "none",
    task_type = "CAUSAL_LM",
)

# === 3. Your Python code (replace this)
python_files = python_files

# === 4. Prepare dataset
dataset = Dataset.from_dict({"text": python_files})

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=2048)

tokenized = dataset.map(tokenize)

# === 5. Set training arguments (small batch for M1)
trainer = FastLanguageModel.get_trainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized,
    args = TrainingArguments(
        output_dir = "deepseek-m1-lora",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        logging_steps = 5,
        learning_rate = 2e-4,
        bf16 = is_bfloat16_supported(),  # Mac M1 supports this
        fp16 = False,
        save_strategy = "epoch",
        optim = "adamw_torch",
    )
)

trainer.train()

# === 6. Save LoRA adapter
model.save_pretrained("deepseek-lora-adapter")
tokenizer.save_pretrained("deepseek-lora-adapter")


/var/folders/hy/36n7jr6x3h74ddx507dc12_00000gn/T/ipykernel_2329/2196248462.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, is_bfloat16_supported


NotImplementedError: Unsloth: No NVIDIA GPU found? Unsloth currently only supports GPUs!

In [64]:
python_files["opensoundscape/audio.py"]

'#!/usr/bin/env python3\n""" audio.py: Utilities for loading and modifying Audio objects\n\n\n**Note: Out-of-place operations**\n\nFunctions that modify Audio (and Spectrogram) objects are "out of place",\nmeaning that they return a new Audio object instead of modifying the\noriginal object. This means that running a line\n```\naudio_object.resample(22050) # WRONG!\n```\nwill **not** change the sample rate of `audio_object`!\nIf your goal was to overwrite `audio_object` with the new,\nresampled audio, you would instead write\n```\naudio_object = audio_object.resample(22050)\n```\n\n"""\nimport warnings\nimport datetime\nfrom pathlib import Path\nimport json\nimport io\nimport urllib\nimport os\n\nimport numpy as np\nimport scipy\n\nimport librosa\nimport librosa.core.audio\nimport soundfile\nimport IPython.display\nfrom aru_metadata_parser.parse import parse_audiomoth_metadata\nfrom aru_metadata_parser.utils import load_metadata\nimport noisereduce\n\nimport opensoundscape\nfrom openso